In [4]:

import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix



class AdaBoost:
    def __init__(self, n_estimators=50):

        self.n_estimators = n_estimators

        self.models = []

        self.alphas = []


    def fit(self, X, y):
        n_samples, n_features = X.shape

        y = np.where(y == 0, -1, 1)

        weights = np.ones(n_samples) / n_samples

        print("Initial Weights:", weights[:5], "...")

        for t in range(self.n_estimators):
            print(f"\nIteration {t+1}")


            model = DecisionTreeClassifier(max_depth=1)
            model.fit(X, y, sample_weight=weights)


            predictions = model.predict(X)

            misclassified = (predictions != y)
            error = np.sum(weights * misclassified)

            error = max(error, 1e-10)

            print("Error:", error)

            alpha = 0.5 * np.log((1 - error) / error)
            print("Alpha:", alpha)

            weights = weights * np.exp(-alpha * y * predictions)

            weights = weights / np.sum(weights)

            print("Updated Weights (first 5):", weights[:5])

            self.models.append(model)
            self.alphas.append(alpha)

    def predict(self, X):
        final_output = np.zeros(X.shape[0])

        for model, alpha in zip(self.models, self.alphas):
            pred = model.predict(X)
            final_output += alpha * pred

        final_pred = np.sign(final_output)

        return np.where(final_pred == -1, 0, 1)


data = load_breast_cancer()
X, y = data.data, data.target

print("Dataset shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

model = AdaBoost(n_estimators=10)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nFinal Accuracy:", accuracy)
print("\nConfusion Matrix:\n", cm)

Dataset shape: (569, 30)
Training samples: 455
Testing samples: 114
Initial Weights: [0.0021978 0.0021978 0.0021978 0.0021978 0.0021978] ...

Iteration 1
Error: 0.07912087912087912
Alpha: 1.227175990733014
Updated Weights (first 5): [0.00119332 0.00119332 0.00119332 0.00119332 0.00119332]

Iteration 2
Error: 0.13540837974012201
Alpha: 0.9269810164029058
Updated Weights (first 5): [0.0006901 0.0006901 0.0006901 0.0006901 0.0006901]

Iteration 3
Error: 0.1612273755032365
Alpha: 0.8245620115557633
Updated Weights (first 5): [0.00041138 0.00041138 0.00041138 0.00214016 0.00041138]

Iteration 4
Error: 0.24367114037994608
Alpha: 0.5663283751032803
Updated Weights (first 5): [0.00027196 0.00027196 0.00027196 0.00141483 0.00027196]

Iteration 5
Error: 0.23621417758581265
Alpha: 0.5867742435499694
Updated Weights (first 5): [0.00017803 0.00017803 0.00017803 0.00299481 0.00017803]

Iteration 6
Error: 0.3229602348304425
Alpha: 0.37010040242469594
Updated Weights (first 5): [0.00013148 0.00013148 